# مقایسهٔ اختیاری: WhisperRT

این نوت‌بوک پیش‌نیاز SURT نیست و وزن یا جزء قابل انتقالی برای SURT تولید نمی‌کند.
برای پژوهش می‌توانید آن را به‌عنوان baseline تک‌گوینده نگه دارید. برای اجرای خود SURT از مرحلهٔ 12 استفاده کنید.
runtime رسمیِ سازگار با `whisper_rt` باید جداگانه نصب و نسخه‌اش ثبت شود؛ صرف دانلود checkpoint کافی نیست.
اگر این مقایسه را نمی‌خواهید، تمام این نوت‌بوک را رد کنید.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/manzourali/English-Streaming-ASR.git"
REPOSITORY_REF = "master"
ROOT = Path("/kaggle/working/English-Streaming-ASR")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_REF, REPOSITORY_URL, str(ROOT)], check=True)
os.chdir(ROOT)
if not (ROOT / "scripts/check_surt_assets.py").is_file():
    raise RuntimeError("This GitHub checkout predates the Kaggle workflow update. Publish the updated source, then start a fresh session.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[kaggle,dev]"], check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
ARTIFACTS = Path("/kaggle/working/asr_artifacts")
ARTIFACTS.mkdir(exist_ok=True)
print("Source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

def run_script(name, *arguments):
    subprocess.run([sys.executable, str(ROOT / "scripts" / name), *map(str, arguments)], check=True)

In [ ]:
RUN_WHISPERRT = False
WHISPERRT_CHECKPOINT = None
CLEAN_MANIFEST = None
if RUN_WHISPERRT:
    import whisper_rt
    from whisper_rt.audio import SpectrogramStream
    from whisper_rt.streaming_decoding import DecodingOptions
    if not WHISPERRT_CHECKPOINT or not CLEAN_MANIFEST:
        raise RuntimeError("Set an actual compatible WhisperRT checkpoint and clean-speech manifest.")
    import yaml
    override = ARTIFACTS / "optional_whisperrt.yaml"
    override.write_text(yaml.safe_dump({
        "data": {"backend": "local", "path": str(CLEAN_MANIFEST)},
        "model": {"checkpoint": str(WHISPERRT_CHECKPOINT)},
        "paths": {"outputs": str(ARTIFACTS / "optional_whisperrt")}
    }, sort_keys=False))
    run_script("run_baseline.py", "--override", override, "--max-samples", 1)
else:
    print("Optional WhisperRT baseline skipped; continue with notebook 12.")